# Price estimation — pipeline walkthrough

Predicting what a product costs from its description, end to end: load the
curated dataset, inspect how items are cleaned, ask a language model for a
price, and score it.

This notebook imports the real modules in `pricer/` rather than redefining
them, so it cannot drift from the code the benchmarks and the live demo use.

It runs on the **free** Groq tier, so executing it costs nothing.

**Setup:** `pip install -r requirements-dev.txt`, then put `GROQ_API_KEY` in `.env`.

In [ ]:
from dotenv import load_dotenv

from pricer.items import Item

load_dotenv(override=True)

# Public dataset of cleaned Amazon listings, curated by ed-donner.
# `items_lite` is the 20k-item split; `items_full` is the complete set.
# It is public, so no Hugging Face token is required.
train, val, test = Item.from_hub("ed-donner/items_lite")
print(f"{len(train):,} train · {len(val):,} validation · {len(test):,} test")

## What one datapoint looks like

Each `Item` carries the raw scraped text (`full`), a condensed `summary`, and
the ground-truth `price`. The summary is what gets fed to the models —
compressing a noisy listing into a few clean lines improves estimates and cuts
token cost at the same time.

In [ ]:
item = test[0]
print(item.title)
print(f"\nactual price: ${item.price:,.2f}\ncategory: {item.category}\n")
print("--- summary given to the model ---")
print(item.summary)

## How the raw data was cleaned

`pricer/parser.py` does the curation: it drops items outside a sane price band,
strips part numbers (which say nothing about price but eat tokens), normalises
weights across five unit systems, and rejects listings too short to be
informative.

In [ ]:
import json

from pricer.parser import MAX_PRICE, MIN_CHARS, MIN_PRICE, get_weight, parse

print(f"keep prices ${MIN_PRICE}-${MAX_PRICE}, descriptions >= {MIN_CHARS} chars\n")

# Weight normalisation, everything converted to pounds.
for raw in ["3.2 ounces", "1.5 pounds", "500 grams", "2 kilograms"]:
    print(f"  {raw:<14} -> {get_weight({'Item Weight': raw}):.3f} lb")

# Out-of-band prices are rejected outright. This sample is deliberately long
# enough to clear the MIN_CHARS floor, so price is the only variable here.
sample = {
    "title": "Example Product",
    "description": ["A durable, well-built product for everyday use."] * 8,
    "features": ["Sturdy construction", "Easy to install"] * 8,
    "details": json.dumps({"Item Weight": "1 pounds"}),
}
print()
for price in ["0.10", "49.99", "5000.00"]:
    kept = parse({**sample, "price": price}, "Electronics")
    print(f"  ${price:>8} -> {'kept' if kept else 'rejected'}")

# Length is a separate gate: a valid price still fails if the listing is thin.
short = {**sample, "price": "49.99", "description": ["Too brief."], "features": []}
print(f"\n  $   49.99 but under the {MIN_CHARS}-char floor -> "
      f"{'kept' if parse(short, 'Electronics') else 'rejected'}")

## Asking a language model for a price

No training involved — just a prompt. The model has to map a description onto a
number, which is a regression task it was never designed for.

One catch worth knowing: `gpt-oss` is a *reasoning* model, and its hidden
reasoning tokens are drawn from the same `max_tokens` budget as the visible
answer. Set that budget too low and the model silently returns an empty string
or a bare `$` — which parses to `$0` and quietly poisons your metrics.

In [ ]:
import os

from groq import Groq

client = Groq(api_key=os.environ["GROQ_API_KEY"])
MODEL = "openai/gpt-oss-20b"

PROMPT = (
    "Estimate the price of this product. Respond with only the price in dollars, "
    "no explanation.\n\n{}"
)


def estimate(item):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": PROMPT.format(item.summary)}],
        max_tokens=200,        # must leave room for hidden reasoning tokens
        reasoning_effort="low",
        temperature=0,
    )
    return response.choices[0].message.content


for it in test[:5]:
    print(f"guess {estimate(it):>8}   actual ${it.price:>8,.2f}   {it.title[:46]}")

## Scoring it properly

`pricer/evaluator.py` runs a predictor across the test set and reports average
absolute error, plus a cumulative-error chart with a 95% confidence band — so
you can see whether the average has actually converged or is still drifting.

Kept small here to stay inside the free tier's 30 requests/minute.

In [ ]:
from pricer.evaluator import evaluate


def gpt_oss_20b(item):
    return estimate(item)


evaluate(gpt_oss_20b, test, size=25, workers=2)

## Full benchmarks

This notebook is a walkthrough, not the measurement. The numbers in the README
come from `benchmarks/`, where every model is scored on an identical slice of
the test set:

```
python benchmarks/run_dnn.py             # 289M-param network, trained locally
python benchmarks/run_openai_prompted.py # GPT-4.1-nano, prompt only
python benchmarks/run_groq.py            # gpt-oss-20b, free tier
```

See `results.ipynb` to chart them.

## On fine-tuning

This project originally included a fine-tuned `gpt-4.1-nano`. That comparison is
no longer reproducible: OpenAI wound down self-serve fine-tuning on
**7 May 2026**, and organisations that had not already run a job lost the ability
to create one. The API now returns `403 training_not_available`
([deprecation notice](https://developers.openai.com/api/docs/deprecations)).

`benchmarks/run_finetune.py` is kept so the attempt stays reproducible.